# 01 · Collect

Downloads raw Steam responses into `data/raw` (reviews, news, store details, online players). Settings are in `configs/config.yaml`.

- **Plan first:** `plan()` asks Steam how many reviews each game × language has (one quick request each), so you see the real size and time before starting.
- **One progress bar for the whole run**, counted in requests, with an ETA.
- **Safe to stop and re-run.** Raw pages are saved as they arrive. Reviews resume where they stopped, and a later run only fetches new and edited reviews; other sources are simply fetched again. `02_parse_clean` deduplicates.
- **Long runs:** the kernel has to stay open. To run without Jupyter open, e.g. overnight in `tmux`:
  `uv run jupyter nbconvert --to notebook --execute --inplace notebooks/01_collect.ipynb`

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging

from src.collect.run import collect, plan
from src.config import CONFIG
from src.db import read
from src.db.connection import init_db

logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
init_db()

## What to collect

Change these to collect less (e.g. a pilot on two games). `None` means the value from `configs/config.yaml`.

In [4]:
SOURCES = "all"   # or e.g. ["reviews"]; all = store_details, reviews, news, player_counts
APPIDS = [275850]     # e.g. [730, 570] or None
PER_LANG = 0  # reviews per game x language; None = config, 0 = all reviews

jobs = plan(SOURCES, APPIDS, PER_LANG)
requests = jobs["requests"].sum()
hours = requests / CONFIG.http.host_rps["store.steampowered.com"] / 3600
print(f"{len(jobs)} jobs, {requests:,} requests, about {hours:.1f} h")
jobs

counting reviews on Steam:   0%|          | 0/7 [00:00<?, ?job/s]

2026-10-31 19:06:36,140 WARNING src.collect.steam_client: HTTP 302 from store.steampowered.com, retry in 30s


KeyboardInterrupt: 

## Run

In [4]:
result = collect(jobs)
result.groupby(["source", "status"]).size()

reviews No Mans Sky [english]:   4%|█                        | 152/3386 [02:30<48:44,  1.11req/s, 1.10 req/s]2026-10-13 14:25:33,133 WARNING src.collect.steam_client: 429 from store.steampowered.com at 1.10 req/s, slowing down to 0.55
2026-10-13 14:25:33,134 WARNING src.collect.steam_client: HTTP 429 from store.steampowered.com, retry in 30s
2026-10-13 14:26:03,956 WARNING src.collect.steam_client: 429 from store.steampowered.com at 0.55 req/s, slowing down to 0.28
2026-10-13 14:26:03,957 WARNING src.collect.steam_client: HTTP 429 from store.steampowered.com, retry in 60s
2026-10-13 14:27:04,593 WARNING src.collect.steam_client: 429 from store.steampowered.com at 0.28 req/s, slowing down to 0.14
2026-10-13 14:27:04,595 WARNING src.collect.steam_client: HTTP 429 from store.steampowered.com, retry in 120s
player_counts No Mans Sky: 100%|██████████████████████████| 2713/2713 [3:01:04<00:00,  4.00s/req, 0.50 req/s]


source         status
news           done      1
player_counts  done      1
reviews        done      4
store_details  done      1
dtype: int64

In [5]:
# Failed jobs (re-run this notebook to retry them)
result[result["status"] == "failed"]

,source,appid,game,language,requests,steam_total,cap,items,status,error


## Review download state

One row per game and language, kept across runs.

In [6]:
read.review_collection()

,appid,language,status,last_cursor,watermark_updated_at,run_max_updated_at,n_items,started_at,finished_at
0,275850,english,done,AoJ4rPX8pOcCfvboXw==,2026-10-13 02:17:15+00:00,2026-10-13 02:17:15+00:00,182483,2026-10-13 11:23:08.870849+00:00,2026-10-13 13:36:49.903541+00:00
1,275850,russian,done,AoJ42sTuv9YCf8TjXg==,2026-10-13 12:34:37+00:00,2026-10-13 12:34:37+00:00,28383,2026-10-13 13:36:49.910329+00:00,2026-10-13 13:52:54.529273+00:00
2,275850,schinese,done,AoJ4/Kic3usCcuWHzAE=,2026-10-13 12:35:53+00:00,2026-10-13 12:35:53+00:00,55498,2026-10-13 13:52:54.535005+00:00,2026-10-13 14:22:13.691262+00:00
3,275850,tchinese,done,AoJwybLvv9YCcMnjXg==,2026-10-12 15:07:13+00:00,2026-10-12 15:07:13+00:00,3032,2026-10-13 14:22:13.697562+00:00,2026-10-13 14:23:50.461981+00:00
4,2357570,english,done,AoJ42r2w8IkDctHFpAQ=,2026-10-12 17:01:48+00:00,2026-10-12 17:01:48+00:00,172612,2026-10-12 18:16:44.865466+00:00,2026-10-12 19:21:33.979758+00:00
5,2357570,russian,done,AoJ4wJay8IkDfuHFpAQ=,2026-10-12 00:14:42+00:00,2026-10-12 00:14:42+00:00,21473,2026-10-12 19:21:33.985340+00:00,2026-10-12 19:29:45.711365+00:00
6,2357570,schinese,done,AoJwh4POiYoDcuPipwQ=,2026-10-11 10:48:37+00:00,2026-10-11 10:48:37+00:00,63583,2026-10-12 19:29:45.718998+00:00,2026-10-12 19:53:50.708186+00:00
7,2357570,tchinese,done,AoJwi4G48IkDd8XGpAQ=,2026-10-11 09:32:31+00:00,2026-10-11 09:32:31+00:00,6117,2026-10-12 19:53:50.713998+00:00,2026-10-12 19:56:10.590449+00:00
8,2379780,english,done,AoJ4492EvpQDc4mVxQU=,2026-10-12 13:38:51+00:00,2026-10-12 13:38:51+00:00,53495,2026-10-12 17:38:36.836301+00:00,2026-10-12 17:56:48.807225+00:00
9,2379780,russian,done,AoJw0JzU440Dfpvq3QQ=,2026-10-12 01:26:01+00:00,2026-10-12 01:26:01+00:00,6074,2026-10-12 17:56:48.812987+00:00,2026-10-12 17:58:52.299381+00:00


Next: `02_parse_clean.ipynb` loads the raw files into Postgres.